# Similarity Search with FAISS
**Goal:** given a query, find the 3 most similar sentences from a small collection.

Steps:
1. Turn every sentence into an **embedding** (a list of numbers that captures its meaning) using `all-MiniLM-L6-v2`.
2. Store the embeddings in a **FAISS index** (a fast search structure).
3. Turn the query into an embedding and ask FAISS for the closest ones.

In [1]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

D:\AI_Demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Our data

In [2]:
sentences = [
    "The cat sat on the sofa and purred.",
    "Dogs are loyal pets that love to play fetch.",
    "Python is a popular programming language for data analysis.",
    "Machine learning models learn patterns from data.",
    "The stock market fell sharply this morning.",
    "I like to cook pasta with fresh tomatoes and basil.",
    "Neural networks are used for image and speech recognition.",
    "The football team won the championship last night.",
    "Investors are worried about rising interest rates.",
    "A kitten was sleeping on the warm blanket.",
]

query = "Which sentences talk about artificial intelligence?"


## Step 1: Generate embeddings
`normalize_embeddings=True` makes every vector length 1, so the inner product between two vectors equals their cosine similarity (1.0 = very similar).

In [3]:
def generate_embeddings(sentences):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(sentences, normalize_embeddings=True)
    embeddings = np.array(embeddings, dtype="float32")  # FAISS needs float32
    return model, embeddings

## Step 2: Create the FAISS index
`IndexFlatIP` compares vectors using the inner product (IP).

In [4]:
def create_index(embeddings):
    dimension = embeddings.shape[1]       # size of each embedding
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)                 # store all embeddings
    return index

## Step 3: Search
Encode the query the same way, then ask the index for the `k` nearest neighbors.

In [5]:
def perform_search(model, query_sentence, index, k):
    query_embedding = model.encode([query_sentence], normalize_embeddings=True)
    query_embedding = np.array(query_embedding, dtype="float32")
    distances, indices = index.search(query_embedding, k)
    return distances, indices

## Run it

In [6]:
model, embeddings = generate_embeddings(sentences)
index = create_index(embeddings)
distances, indices = perform_search(model, query, index, k=3)

print("Query:", query)
print()
for rank in range(3):
    sentence = sentences[indices[0][rank]]
    score = distances[0][rank]
    print(f"{rank + 1}. {sentence}  (similarity: {score:.4f})")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 33173.10it/s]


Query: Which sentences talk about artificial intelligence?

1. Machine learning models learn patterns from data.  (similarity: 0.3169)
2. Neural networks are used for image and speech recognition.  (similarity: 0.2881)
3. Python is a popular programming language for data analysis.  (similarity: 0.1499)


## Try it yourself
- Change `query` to something about animals, money or food. Do the top 3 change as you expect?
- Change `k` to 5.
- Add your own sentences to the list.